<a href="https://colab.research.google.com/github/zeinafarghaly-arch/ML-flyrank/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — "What Predicts Health?" (Random Forest, ML Appendix)

The paper says that a Random Forest model was used to predict `health_score`. The most important features were Average Position (43%), Impressions (32%), and Scroll Depth (15%). The model was tested on a separate set of data.

**My question about the method**

The paper explains that `health_score` is calculated using Impressions, Position, CTR, and Scroll Depth. These are the same features that the model says are the most important.

The paper already says that these results are "descriptive rather than causal." I would also point out that testing the model on a separate dataset does not change the fact that the score is already made from these same features.

Because of this, the model is mainly learning the formula used to create the score. It is not finding new factors that predict health. It may be clearer to say that the model "recovered the known formula" instead of saying it "discovered what predicts health." This would help readers understand that the result comes from the way the score was created.


### Finding 2 — "What Predicts Growth?" (Logistic Regression, 71% holdout accuracy)

The paper says that a logistic regression model was used to predict whether a page would grow or decline. The model reached 71% accuracy on the test data, and content age had the strongest negative effect.

**My question about the method**

The paper says the model used an 80/20 train-test split, but it does not say whether the split was done by brand. Since the dataset includes 57 brands, pages from the same brand may have similar characteristics, such as domain authority, topics, and publishing style.

If pages from the same brand were included in both the training and test sets, the model might partly learn patterns that are specific to each brand instead of learning general patterns that apply to all brands. This could make the reported 71% accuracy look higher than it really is.

Also, the paper does not say how many pages were growing and how many were declining. Without this information, it is difficult to know how meaningful the 71% accuracy is.

I would like to ask whether the train-test split was grouped by brand, and what the percentage of growing versus declining pages was. This would help readers better understand how strong the model's results really are.


In [1]:
import pandas as pd

paper_facts = pd.DataFrame([
    {"finding": "What Predicts Health? (Random Forest)",
     "label": "health_score = impressions(30) + position(30) + ctr(20) + scroll_depth(20)",
     "top_features_reported": "avg_position 43%, impressions 32%, scroll_depth 15%, ctr 8%",
     "disclosed_split": "80/20 (row-level, not stated as grouped)",
     "my_concern": "label is a formula built from its own top 4 reported features"},
    {"finding": "What Predicts Growth? (Logistic Regression)",
     "label": "growing vs declining, from 30d-vs-prev-30d impression trend",
     "top_features_reported": "content_age (neg), days_since_update (neg), days_visible (pos)",
     "disclosed_split": "80/20 (row-level, not stated as grouped)",
     "my_concern": "57 brands + row-level split -> possible brand leakage; base rate not disclosed"},
])
paper_facts

,finding,label,top_features_reported,disclosed_split,my_concern
0,What Predicts Health? (Random Forest),health_score = impressions(30) + position(30) ...,"avg_position 43%, impressions 32%, scroll_dept...","80/20 (row-level, not stated as grouped)",label is a formula built from its own top 4 re...
1,What Predicts Growth? (Logistic Regression),"growing vs declining, from 30d-vs-prev-30d imp...","content_age (neg), days_since_update (neg), da...","80/20 (row-level, not stated as grouped)",57 brands + row-level split -> possible brand ...


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

My Week-5 model was already trained with a `GroupShuffleSplit` grouped by `client_id`, so the honest split isn't new here -- what's new is proving it mattered. To show the before/after, I re-ran the exact same pipeline (6 features, `LogisticRegression`, same baseline rule) under a **naive random 80/25 row-level split** -- the split I would have used if I hadn't thought about it -- and compare it to my **honest client-grouped split**.


In [3]:
from google.colab import files
uploaded = files.upload()

import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit

df = pd.read_csv("content_refresh_anonymized.csv")

print("Dataset loaded successfully!")
print(f"Rows: {len(df)}")
print(f"Brands: {df['client_id'].nunique()}")

df["is_declining"] = (df["trend_direction"] == "down").astype(int)

declining = df["is_declining"].mean() * 100

print("\nClass Balance")
print(f"Declining pages: {declining:.1f}%")
print(f"Growing pages: {100 - declining:.1f}%")

train_random, test_random = train_test_split(
    df,
    test_size=0.25,
    random_state=42,
    stratify=df["is_declining"]
)

random_overlap = len(
    set(train_random["client_id"]) &
    set(test_random["client_id"])
)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df, groups=df["client_id"])
)

train_group = df.iloc[train_idx]
test_group = df.iloc[test_idx]

group_overlap = len(
    set(train_group["client_id"]) &
    set(test_group["client_id"])
)

print("\n===== RESULTS =====")

print("\nRandom Split")
print(f"Brands appearing in BOTH train and test: {random_overlap}")

print("\nGrouped Split")
print(f"Brands appearing in BOTH train and test: {group_overlap}")

if random_overlap > 0:
    print("\n The random split allows brand leakage.")
else:
    print("\n No brand leakage in the random split.")

if group_overlap == 0:
    print(" The grouped split completely prevents brand leakage.")
else:
    print(" Some brands still overlap.")

Saving content_refresh_anonymized.csv to content_refresh_anonymized.csv
Dataset loaded successfully!
Rows: 30000
Brands: 32

Class Balance
Declining pages: 54.2%
Growing pages: 45.8%

===== RESULTS =====

Random Split
Brands appearing in BOTH train and test: 31

Grouped Split
Brands appearing in BOTH train and test: 0

 The random split allows brand leakage.
 The grouped split completely prevents brand leakage.


**Observed, not celebrated:** In the random split, 31 of the 32 clients appear in both the training and test sets. This means the model is partly tested on clients it has already seen, so some of its performance may come from recognizing client-specific patterns instead of learning general ones. In the grouped split, the training and test sets contain different clients, so the model is tested on completely unseen clients. As expected, the performance is lower (baseline: 0.490, model: 0.528).

**Two things I learned:**

* The model performs a little better than the baseline in both cases, but the improvement is much smaller when using the grouped split. The gap drops from about **0.09** with the random split to about **0.04** with the grouped split. I will report the **0.04** result because it gives a more realistic picture of the model's performance.

* The model's Precision@500 on the grouped split is **0.528**, which is close to the overall base rate of **0.542**. This means the model is only slightly better than choosing pages at random. It may still be useful as a decision-support tool, but the improvement is small, so I should describe the result honestly instead of focusing on the higher number from the random split.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Attacking My Own Model

To make sure my results are trustworthy, I checked my model for common sources of data leakage using the honest client-grouped split.

* All six features (`days_since_last_update`, `search_volume`, `avg_position`, `ctr`, `engagement_rate`, and `content_age_days`) are available at the time the prediction is made. The model predicts whether the page will decline over the next 30 days.
*  I checked that none of the features were created from the label or closely related to it.
*  I did not use `health_score` or any optimization flags as model features.
*  The train-test split was grouped by `client_id`, so the same client never appears in both sets.
*  I reported the base rate (0.542) alongside the model results.
*  All results were measured on the held-out test set, not on the training data.

To make sure these checks really work, I added the excluded features back into the model one at a time and watched how Precision@500 changed. If the score increases a lot after adding one of these features, it suggests that the feature is leaking information that the model should not have during prediction.


In [6]:
# Self-contained: re-imports and re-builds the honest split so this cell runs
# correctly even if Section 2 wasn't executed first in this kernel session.
import pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

df = pd.read_csv("content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)
features = ["days_since_last_update", "search_volume", "avg_position",
            "ctr", "engagement_rate", "content_age_days"]

def precision_at_k(scores, labels, k=500):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_idx, te_idx = next(splitter.split(df, groups=df["client_id"]))
grp_train, grp_test = df.iloc[tr_idx].copy(), df.iloc[te_idx].copy()

def fit_and_p500(trial_features):
    X_train, y_train = grp_train[trial_features], grp_train["is_declining"]
    X_test, y_test = grp_test[trial_features], grp_test["is_declining"]
    trial_model = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(class_weight="balanced", random_state=42)),
    ])
    trial_model.fit(X_train, y_train)
    scores = trial_model.predict_proba(X_test)[:, 1]
    return precision_at_k(scores, y_test.values)

suspect_features = ["impressions_last_30d", "impressions_prev_30d", "trend_pct"]

honest_p500 = fit_and_p500(features)
print(f"Honest 6-feature model, Precision@500: {honest_p500:.3f}\n")

audit_rows = []
for suspect in suspect_features:
    trial_p500 = fit_and_p500(features + [suspect])
    audit_rows.append({
        "added_feature": suspect,
        "Precision@500_with_suspect": round(trial_p500, 3),
        "jump_vs_honest": round(trial_p500 - honest_p500, 3),
    })

pd.DataFrame(audit_rows)

Honest 6-feature model, Precision@500: 0.528



,added_feature,Precision@500_with_suspect,jump_vs_honest
0,impressions_last_30d,0.548,0.020
1,impressions_prev_30d,0.528,0.000
2,trend_pct,1.000,0.472


**Observed:** When I added `trend_pct` back into the model, Precision@500 increased to **1.000**. This large jump shows that `trend_pct` gives the model information that is almost the same as the label. Since `is_declining` is created from `trend_direction`, and `trend_direction` comes from `trend_pct`, this feature is effectively revealing the answer to the model.

Adding `impressions_last_30d` and `impressions_prev_30d` had only a small effect (+0.02 and +0.00). These features contain some related information, but they do not directly reveal the label like `trend_pct` does.

These results confirm that excluding all three features was the correct decision. Because of this, I will report the honest Precision@500 of **0.528**, as it better reflects how the model performs on unseen clients.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [7]:
claim_pairs = [
    {
        "original": "Logistic Regression beat the rule-based baseline on Precision@500.",
        "rewritten": ("Under an honest, client-grouped holdout, the logistic regression model measured "
                      "a Precision@500 of 0.528 versus 0.490 for the rule-based baseline -- a modest, "
                      "directional improvement (~4 points) that supports using it as a decision-support "
                      "shortlist, not as a settled replacement for the rule."),
    },
    {
        "original": ("The model mostly leans on days_since_last_update and avg_position -- both make sense: "
                     "stale, poorly-ranked pages are more likely to be declining."),
        "rewritten": ("In this sample, days_since_last_update and avg_position carried the largest "
                      "coefficient weight in the model. That is consistent with -- but does not prove -- "
                      "the intuition that staleness and poor ranking drive decline; it is an observed "
                      "association in this holdout, not a causal claim."),
    },
]

for pair in claim_pairs:
    print("ORIGINAL: ", pair["original"])
    print("REWRITTEN:", pair["rewritten"])
    print()

ORIGINAL:  Logistic Regression beat the rule-based baseline on Precision@500.
REWRITTEN: Under an honest, client-grouped holdout, the logistic regression model measured a Precision@500 of 0.528 versus 0.490 for the rule-based baseline -- a modest, directional improvement (~4 points) that supports using it as a decision-support shortlist, not as a settled replacement for the rule.

ORIGINAL:  The model mostly leans on days_since_last_update and avg_position -- both make sense: stale, poorly-ranked pages are more likely to be declining.
REWRITTEN: In this sample, days_since_last_update and avg_position carried the largest coefficient weight in the model. That is consistent with -- but does not prove -- the intuition that staleness and poor ranking drive decline; it is an observed association in this holdout, not a causal claim.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.